# Calculate Risk Novelty Scores (Cosine Similarity)

This notebook calculates the "Novelty Score" for risk disclosure texts.
We compare the risk text of the current year ($t$) with the previous year ($t-1$) for each company.

## Methodology
1. **Embedding**: Use a Japanese SBERT model to convert risk texts into semantic vectors.
2. **Cosine Similarity**: Calculate the similarity between $Vector_t$ and $Vector_{t-1}$.
3. **Novelty Score**: Defined as $1 - CosineSimilarity$.

## Output
- Saves the scores to the `risk_novelty_scores` table in PostgreSQL.

In [15]:
import os
import sys
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Set project root
os.chdir(os.path.abspath(".."))
sys.path.append(os.path.abspath(".."))

load_dotenv()

# Database Connection
DB_USER = os.getenv("POSTGRES_USER")
DB_PASS = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB")

db_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)
print("Connected to DB")

Connected to DB


## 1. Load Risk Texts

In [16]:
query = """
SELECT 
    doc_id, 
    company_id, 
    fiscal_year, 
    risk_text
FROM edinet_documents
WHERE risk_text IS NOT NULL AND risk_text != '';
"""
df_docs = pd.read_sql(query, engine)
print(f"Loaded {len(df_docs)} documents")

# --- Text Cleaning ---
import re

def clean_risk_text(text):
    if not text:
        return ""
    
    # 1. Remove Headers (e.g., "3【事業等のリスク】", "(1) 経営成績等の状況")
    # Remove patterns like 【...】 or [number] at the start of lines or paragraphs
    text = re.sub(r'【.*?】', '', text)
    text = re.sub(r'第[０-９0-9]+', '', text)
    
    # 2. Normalize Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Cleaning text...")
df_docs['risk_text'] = df_docs['risk_text'].apply(clean_risk_text)
print("Text cleaning done.")

df_docs.head()

Loaded 8607 documents
Cleaning text...
Text cleaning done.


,doc_id,company_id,fiscal_year,risk_text
0,S100W5DK,1cda0461-2faf-4a21-ac31-ef683d997bba,2025,３ ⑴当社のリスクマネジメント体制当社は、当社グループ内で発生し得る様々なリスクに対し、発生...
1,S100TL6G,c79cba27-b7b4-4843-8fbb-ed92f665deb2,2024,3 1. リスク管理体制当社では、部門・営業グループと各リスクに対応したコーポレート専門部局...
2,S100VU5K,b4e96dbc-fc77-418d-bfb8-716519511400,2025,３ 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が当企業グルー...
3,S100VJ78,8002d140-1dfe-4968-b672-6f4bc311c8f9,2025,3 当社グループの事業その他に関するリスクについて、投資家の判断に重要な影響を及ぼす可能性が...
4,S100W4TI,e9c56d9c-7563-42ab-86cf-bf6eed25a9fe,2025,３ 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が連結会社の財...


## 2. Prepare Pairs (t vs t-1)

In [17]:
# Self-join to match current year (t) with previous year (t-1)
df_docs['prev_year'] = df_docs['fiscal_year'] - 1

df_pairs = pd.merge(
    df_docs, 
    df_docs[['company_id', 'fiscal_year', 'risk_text', 'doc_id']], 
    left_on=['company_id', 'prev_year'], 
    right_on=['company_id', 'fiscal_year'], 
    suffixes=('_curr', '_prev'),
    how='inner'
)

# Filter relevant columns
df_pairs = df_pairs[[
    'company_id', 
    'fiscal_year_curr', 
    'doc_id_curr', 
    'doc_id_prev',
    'risk_text_curr', 
    'risk_text_prev'
]]

print(f"Created {len(df_pairs)} pairs for comparison")
df_pairs.head()

Created 7345 pairs for comparison


,company_id,fiscal_year_curr,doc_id_curr,doc_id_prev,risk_text_curr,risk_text_prev
0,1cda0461-2faf-4a21-ac31-ef683d997bba,2025,S100W5DK,S100TOSN,３ ⑴当社のリスクマネジメント体制当社は、当社グループ内で発生し得る様々なリスクに対し、発生...,３ ⑴当社のリスクマネジメント体制当社は、当社グループ内で発生しうる様々なリスクに対し、発生...
1,c79cba27-b7b4-4843-8fbb-ed92f665deb2,2024,S100TL6G,S100QVYU,3 1. リスク管理体制当社では、部門・営業グループと各リスクに対応したコーポレート専門部局...,3 1. リスク管理体制当社では、部門・営業グループと各リスクに対応したコーポレート専門部局...
2,b4e96dbc-fc77-418d-bfb8-716519511400,2025,S100VU5K,S100TIMV,３ 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が当企業グルー...,３ 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が当企業グルー...
3,8002d140-1dfe-4968-b672-6f4bc311c8f9,2025,S100VJ78,S100T6GJ,3 当社グループの事業その他に関するリスクについて、投資家の判断に重要な影響を及ぼす可能性が...,3 当社グループの事業その他に関するリスクについて、投資家の判断に重要な影響を及ぼす可能性が...
4,e9c56d9c-7563-42ab-86cf-bf6eed25a9fe,2025,S100W4TI,S100TV82,３ 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が連結会社の財...,３ 有価証券報告書に記載した事業の状況、経理の状況等に関する事項のうち、経営者が連結会社の財...


## 3. Calculate Cosine Similarity with SBERT

In [20]:
# Install sentence-transformers if not installed
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer, util

# Load Japanese SBERT model
# Using a lightweight model for efficiency: 'pkshatech/GLuCoSE-base-ja' is good but might be heavy.
# Let's use a standard multilingual or japanese specific one.
model_name = "intfloat/multilingual-e5-small" # or "sonoisa/sentence-bert-base-ja-mean-tokens"
print(f"Loading model: {model_name}...")
model = SentenceTransformer(model_name)
print("Model loaded.")

Loading model: intfloat/multilingual-e5-small...
Model loaded.


In [21]:
# Function to calculate similarity in batches
def calculate_similarity_batch(pairs_df, batch_size=32):
    similarities = []
    
    # Split into batches
    for i in tqdm(range(0, len(pairs_df), batch_size), desc="Calculating Similarity"):
        batch = pairs_df.iloc[i:i+batch_size]
        
        # Prepare texts (add prefix for E5 model if used)
        texts_curr = ["query: " + t for t in batch['risk_text_curr'].tolist()] 
        texts_prev = ["query: " + t for t in batch['risk_text_prev'].tolist()]
        
        # Encode
        emb_curr = model.encode(texts_curr, convert_to_tensor=True, normalize_embeddings=True)
        emb_prev = model.encode(texts_prev, convert_to_tensor=True, normalize_embeddings=True)
        
        # Compute cosine similarity (diagonal elements)
        # emb_curr and emb_prev are (batch_size, hidden_dim)
        # We want row-wise cosine similarity
        cosine_scores = (emb_curr * emb_prev).sum(dim=1).cpu().numpy()
        similarities.extend(cosine_scores)
        
    return similarities

# Run calculation
# For testing, let's run on a sample first
# sample_df = df_pairs.head(100)
# sims = calculate_similarity_batch(sample_df)

# Run on full dataset
df_pairs['cosine_similarity'] = calculate_similarity_batch(df_pairs)
df_pairs['novelty_score'] = 1 - df_pairs['cosine_similarity']

df_pairs[['company_id', 'fiscal_year_curr', 'cosine_similarity', 'novelty_score']].head()

Calculating Similarity:   0%|          | 0/230 [00:00<?, ?it/s]

,company_id,fiscal_year_curr,cosine_similarity,novelty_score
0,1cda0461-2faf-4a21-ac31-ef683d997bba,2025,0.999452,5.482435e-04
1,c79cba27-b7b4-4843-8fbb-ed92f665deb2,2024,0.988194,1.180619e-02
2,b4e96dbc-fc77-418d-bfb8-716519511400,2025,0.999973,2.676249e-05
3,8002d140-1dfe-4968-b672-6f4bc311c8f9,2025,1.000000,-1.192093e-07
4,e9c56d9c-7563-42ab-86cf-bf6eed25a9fe,2025,1.000000,0.000000e+00


## 4. Save Scores to Database

In [12]:
# Create table if not exists
create_table_sql = """
CREATE TABLE IF NOT EXISTS risk_novelty_scores (
    company_id UUID,
    fiscal_year INT,
    doc_id_curr VARCHAR(20),
    doc_id_prev VARCHAR(20),
    cosine_similarity FLOAT,
    novelty_score FLOAT,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (company_id, fiscal_year)
);
"""
with engine.begin() as conn:
    conn.execute(text(create_table_sql))

# Prepare data for insert
insert_data = df_pairs[[
    'company_id', 
    'fiscal_year_curr', 
    'doc_id_curr', 
    'doc_id_prev', 
    'cosine_similarity', 
    'novelty_score'
]].rename(columns={'fiscal_year_curr': 'fiscal_year'})

# Upsert
from sqlalchemy.dialects.postgresql import insert

def upsert_scores(df, engine):
    records = df.to_dict(orient='records')
    
    stmt = insert(text("risk_novelty_scores")).values(records)
    stmt = stmt.on_conflict_do_update(
        index_elements=['company_id', 'fiscal_year'],
        set_={
            'cosine_similarity': stmt.excluded.cosine_similarity,
            'novelty_score': stmt.excluded.novelty_score,
            'updated_at': text('CURRENT_TIMESTAMP')
        }
    )
    
    with engine.begin() as conn:
        conn.execute(stmt)
        
# Note: Standard to_sql might fail on upsert, better to use custom or loop if data is huge.
# For ~8000 records, pure pandas to_sql with replacement or manual batch insert is fine.
# Here we use a simple replacement for simplicity first.

with engine.begin() as conn:
    # Clear existing data for safety or use upsert logic above
    # conn.execute(text("TRUNCATE TABLE risk_novelty_scores"))
    pass

# Using pandas to_sql (replace mode for now, simpler)
# Be careful not to drop the table schema if it has other constraints
insert_data.to_sql('risk_novelty_scores', engine, if_exists='replace', index=False, method='multi', chunksize=1000)
print("Saved scores to database.")

Saved scores to database.
